# Lab 4: Campaign Analytics MCP Tools with AgentCore Gateway

## Overview

In this advanced lab, you'll learn how to transform AWS Lambda functions into Model Context Protocol (MCP) tools using Amazon Bedrock AgentCore Gateway. This creates a secure, scalable foundation for AI agents to access campaign analytics capabilities through standardized interfaces.

### What You'll Build

**Campaign Analytics MCP Tools** that provide:

- **Player Context Retrieval**: SQL-based player segmentation with comprehensive data aggregation
- **Semantic Item Recommendations**: S3 Vectors-powered item discovery for targeted campaigns
- **Dynamic Pricing Calculator**: Flexible pricing strategy computation with multiple discount models
- **Secure Gateway Access**: Cognito JWT authentication with IAM-based outbound authorization
- **Production-Ready Architecture**: Scalable Lambda functions with proper error handling and logging

## Prerequisites

To execute this tutorial you will need:
* **Lab 1 completed** - S3 Vectors infrastructure with gaming datasets indexed
* **Lab 2 completed** - Understanding of player segmentation and vector queries
* AWS credentials configured with appropriate permissions
* Amazon Bedrock AgentCore Gateway access
* Amazon Cognito access for authentication setup

> **How to run:** Click each code cell and press Shift+Enter or the Run button. Execute cells in order from top to bottom.

## Step 1: Environment Setup and Dependencies

This section installs all required packages and sets up the development environment. We're installing the Strands Agents framework for AI agent development, MCP client libraries for protocol communication, and various AWS SDKs for service integration.

The setup also configures our AWS region and imports the gateway utilities that will handle Lambda function creation, IAM role management, and Cognito authentication setup.

In [ ]:
# Install required packages
!pip install boto3 botocore strands-agents "mcp[cli]" requests --q
print("✅ Dependencies installed successfully")

### Import Libraries and Setup Environment

Import the gateway utilities and configure AWS region settings for AgentCore Gateway deployment.

In [ ]:
# Import utilities and setup
import os
import sys
import boto3

# Add utils to path
current_dir = os.getcwd()
utils_dir = os.path.join(current_dir, 'utils')
sys.path.insert(0, utils_dir)

import utils.gateway_utils as gateway_utils

# Get AWS region from environment or use default
REGION = os.environ.get('AWS_DEFAULT_REGION') or boto3.Session().region_name or 'us-west-2'
print(f"Using AWS region: {REGION}")

## Step 2: Create Campaign Analytics Lambda Function

This section creates the core Lambda function that serves as our MCP tools backend. The [Model Context Protocol (MCP)](https://modelcontextprotocol.io/docs/getting-started/intro) is an open-source standard for connecting AI applications to external systems. Using MCP, AI applications can connect to data sources, tools, and workflows—enabling them to access key information and perform tasks. Think of MCP like a USB-C port for AI applications.

The function implements three essential campaign analytics capabilities: SQL-based player context retrieval, semantic item recommendations using S3 Vectors, and dynamic pricing calculations with multiple discount models.

In [ ]:
%%writefile game_analytics_lambda.py
import json
import boto3
import numpy as np
from datetime import datetime

# Configuration
VECTOR_BUCKET = "game-embeddings"

s3_client = boto3.client('s3')
bedrock = boto3.client('bedrock-runtime')
s3vectors = boto3.client('s3vectors')

### Query Embedding Helper Function

This helper function generates optimized embeddings for search queries using Cohere's embed-english-v3 model. It uses the `search_query` input type for better semantic matching and includes proper error handling and numpy float32 conversion for optimal S3 Vectors compatibility.

In [ ]:
%%writefile -a game_analytics_lambda.py

def generate_query_embedding(text):
    """Generate embedding optimized for search queries"""
    if not text or not text.strip():
        raise ValueError("Query text cannot be empty")
    
    try:
        response = bedrock.invoke_model(
            modelId="cohere.embed-english-v3",
            body=json.dumps({
                "texts": [text],
                "input_type": "search_query"
            })
        )
        embedding = json.loads(response["body"].read())["embeddings"][0]
        return np.array(embedding, dtype=np.float32).tolist()
    except Exception as e:
        raise Exception(f"Failed to generate query embedding: {str(e)}")

### Player Context By Criteria Retrieval Function

This function executes SQL queries against player datasets and retrieves comprehensive player profiles using S3 Vectors. It combines structured SQL filtering with vector-based data retrieval to provide rich context for campaign targeting decisions.

In [ ]:
%%writefile -a game_analytics_lambda.py

def get_player_context_by_criteria(event):
    criteria = event.get('criteria', '')
    datasets = ["player_profiles", "lifetime_spend", "session_history", "player_attributes"]
    
    if not criteria:
        return {'error': 'criteria is required'}
    
    try:
        # Import s3_query utility
        import sys
        import os
        import boto3
        sys.path.append('/opt/python/lib/python3.9/site-packages')  # Lambda layer path
        from utils.s3_query import execute_s3_select_query
        
        # Get bucket name from SSM parameter
        ssm = boto3.client('ssm')
        bucket_name = ssm.get_parameter(Name='/workshop/games-data-bucket-name')['Parameter']['Value']
        
        # Execute the criteria query to get player_ids
        player_ids = execute_s3_select_query(criteria, bucket_name)
        if not player_ids:
            return {'error': 'No players found matching criteria'}

        player_context = {}
        
        # Search each dataset for the specified player IDs using S3 Vectors
        for dataset in datasets:
            try:
                # Use S3 Vectors get_vectors API to retrieve specific player data
                response = s3vectors.get_vectors(
                    vectorBucketName="game-embeddings",
                    indexName=dataset.replace('_', '-'),
                    keys=player_ids,
                    returnMetadata=True
                )
                
                # Build comprehensive player profiles from the response
                for vector in response.get('vectors', []):
                    player_id = vector.get('key')
                    if player_id in player_ids:
                        if player_id not in player_context:
                            player_context[player_id] = {}
                        player_context[player_id][dataset] = vector.get('metadata', {})
            except Exception as e:
                print(f"Failed to query {dataset}: {str(e)}")
                continue
        
        return {
            'criteria': criteria,
            'player_context': player_context,
            'players_found': len(player_context),
        }
        
    except Exception as e:
        return {'error': f'Context retrieval failed: {str(e)}'}

### Semantic Item Recommendation Function

This function performs semantic search across the game items catalog using natural language queries. It generates query embeddings and uses S3 Vectors to find the most relevant items based on semantic similarity, returning detailed item metadata with relevance scores.

In [ ]:
%%writefile -a game_analytics_lambda.py

def recommend_items_for_segment(event):
    query_text = event.get('query_text', '')
    campaign_goal = event.get('campaign_goal', 'monetization')
    
    if not query_text:
        return {'error': 'query_text is required'}
    
    try:
        # Generate embedding using the optimized function
        query_embedding = generate_query_embedding(query_text)
        
        # Search items using S3 Vectors semantic search
        response = s3vectors.query_vectors(
            vectorBucketName="game-embeddings",
            indexName='items',
            queryVector={"float32": query_embedding},
            topK=10,
            returnDistance=True,
            returnMetadata=True
        )
        
        recommendations = []
        for vector in response.get('vectors', []):
            item_data = vector.get('metadata', {})
            recommendations.append({
                'item_id': item_data.get('item_id', vector.get('key')),
                'name': item_data.get('name', ''),
                'category': item_data.get('category', ''),
                'base_price': item_data.get('base_price', 0),
                'rarity': item_data.get('rarity', ''),
                'faction_affinity': item_data.get('faction_affinity'),
                'player_level_req': item_data.get('player_level_req', 0),
                'relevance_score': vector.get('score', 0)
            })
        
        return {
            'campaign_goal': campaign_goal,
            'search_query': query_text,
            'recommended_items': recommendations
        }
    
    except Exception as e:
        return {'error': f'Item recommendation failed: {str(e)}'}

### Dynamic Pricing Strategy Calculator

This function applies flexible pricing strategies to items based on campaign requirements. It supports multiple pricing models including percentage discounts, markups, and fixed pricing, enabling AI agents to dynamically adjust pricing based on market conditions and campaign goals.

In [ ]:
%%writefile -a game_analytics_lambda.py

def calculate_pricing_strategy(event):
    items = event.get('items', [])
    pricing_instructions = event.get('pricing_instructions', [])
    
    if not items or not pricing_instructions:
        return {'error': 'items and pricing_instructions are required'}
    
    try:
        priced_items = []
        
        for i, item in enumerate(items):
            base_price = item.get('base_price', 0)
            
            # Get pricing instruction for this item (or use last one if fewer instructions)
            instruction = pricing_instructions[min(i, len(pricing_instructions) - 1)]
            
            if instruction.get('type') == 'discount':
                discount_percent = instruction.get('value', 0)
                final_price = round(base_price * (1 - discount_percent/100), 2)
            
            elif instruction.get('type') == 'markup':
                markup_percent = instruction.get('value', 0)
                final_price = round(base_price * (1 + markup_percent/100), 2)
            
            elif instruction.get('type') == 'fixed_price':
                final_price = instruction.get('value', base_price)
            
            else:
                final_price = base_price
            
            priced_items.append({
                'item_id': item.get('item_id'),
                'name': item.get('name'),
                'base_price': base_price,
                'final_price': final_price
            })
        
        return {
            'priced_items': priced_items
        }
    
    except Exception as e:
        return {'error': f'Pricing calculation failed: {str(e)}'}

### Lambda Handler Function

Finally, we setup the function handler that serves as the entry point for all MCP tool calls from [Amazon Bedrock AgentCore Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html). Gateway provides an easy and secure way to convert Lambda functions into Model Context Protocol (MCP)-compatible tools, handling authentication, protocol translation, and tool discovery automatically. 

It extracts the tool name from the context, routes requests to the appropriate tool function, and handles responses with proper error handling and status codes.

In [ ]:
%%writefile -a game_analytics_lambda.py

def lambda_handler(event, context):
    # Extract tool name from context (AgentCore Gateway provides this)
    tool_name = context.client_context.custom['bedrockAgentCoreToolName']
    print(context.client_context)
    print(event)
    print(f"Original toolName: {tool_name}")
    
    # Remove target prefix if present (format: targetname___toolname)
    delimiter = "___"
    if delimiter in tool_name:
        tool_name = tool_name[tool_name.index(delimiter) + len(delimiter):]
    print(f"Converted toolName: {tool_name}")
    
    tool_functions = {
        'get_player_context_by_criteria': get_player_context_by_criteria,
        'recommend_items_for_segment': recommend_items_for_segment,
        'calculate_pricing_strategy': calculate_pricing_strategy
    } 
    
    if tool_name in tool_functions:
        try:
            result = tool_functions[tool_name](event)
            return {'statusCode': 200, 'body': json.dumps(result)}
        except Exception as e:
            return {'statusCode': 500, 'body': json.dumps({'error': str(e)})}
    else:
        return {'statusCode': 404, 'body': json.dumps({'error': f'Tool {tool_name} not found'})}

### Lambda Function Deployment

This cell deploys our Lambda function using the gateway utilities. The deployment process creates an IAM role with necessary permissions, packages the code with dependencies, and configures the Python 3.11 runtime with appropriate timeout and memory settings.

In [ ]:
# Create Lambda function using utility
lambda_resp = gateway_utils.create_gateway_lambda("game_analytics_lambda.py")

if lambda_resp['exit_code'] == 0:
    print("Lambda function created with ARN:", lambda_resp['lambda_function_arn'])
    lambda_function_arn = lambda_resp['lambda_function_arn']
else:
    print("Lambda function creation failed")

## Step 3: Create IAM Role for AgentCore Gateway

This section creates a comprehensive IAM role that enables AgentCore Gateway to securely access AWS resources on behalf of authenticated users. The role implements the outbound authentication component of the dual authentication model, acting as a secure proxy between AI agents and AWS services.

In [ ]:
# Create IAM role for AgentCore Gateway using utility
agentcore_gateway_iam_role = gateway_utils.create_agentcore_gateway_role("game-analytics")
print("AgentCore gateway role ARN:", agentcore_gateway_iam_role['Role']['Arn'])
gateway_role_arn = agentcore_gateway_iam_role['Role']['Arn']

## Step 4: Setup Amazon Cognito for Inbound Authentication

This section implements the inbound authentication component using Amazon Cognito OAuth 2.0 infrastructure. We create a user pool, domain, resource server with granular scopes, and machine-to-machine client to ensure only authenticated and authorized clients can access our campaign analytics tools.

In [ ]:
# Cognito configuration
USER_POOL_NAME = "game-analytics-gateway-pool"
RESOURCE_SERVER_ID = "game-analytics-gateway-id"
RESOURCE_SERVER_NAME = "game-analytics-gateway-name"
CLIENT_NAME = "game-analytics-gateway-client"
SCOPES = [
    {"ScopeName": "gateway:read", "ScopeDescription": "Read access to game analytics"},
    {"ScopeName": "gateway:write", "ScopeDescription": "Write access to game analytics"}
]
scopeString = f"{RESOURCE_SERVER_ID}/gateway:read {RESOURCE_SERVER_ID}/gateway:write"

cognito = boto3.client("cognito-idp", region_name=REGION)

print("Creating or retrieving Cognito resources...")
user_pool_id = gateway_utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
print(f"User Pool ID: {user_pool_id}")

gateway_utils.get_or_create_resource_server(cognito, user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES)
print("Resource server ensured.")

client_id, client_secret = gateway_utils.get_or_create_m2m_client(cognito, user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID)
print(f"Client ID: {client_id}")

cognito_discovery_url = f'https://cognito-idp.{REGION}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration'
print(f"Cognito Discovery URL: {cognito_discovery_url}")

## Step 5: Create AgentCore Gateway

This section creates the AgentCore Gateway that serves as the central hub for our MCP tools. The gateway acts as a secure, protocol-compliant server that exposes our Lambda functions as standardized MCP tools with dual authentication using inbound JWT validation and outbound IAM authorization.

In [ ]:
# Create AgentCore Gateway
gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)

auth_config = {
    "customJWTAuthorizer": { 
        "allowedClients": [client_id],
        "discoveryUrl": cognito_discovery_url
    }
}

create_response = gateway_client.create_gateway(
    name='game-analytics-gateway',
    roleArn=gateway_role_arn,
    protocolType='MCP',
    authorizerType='CUSTOM_JWT',
    authorizerConfiguration=auth_config, 
    description='Game Analytics Gateway with comprehensive tools'
)

print("Gateway created successfully:")
gatewayID = create_response["gatewayId"]
gatewayURL = create_response["gatewayUrl"]
print(f"Gateway ID: {gatewayID}")
print(f"Gateway URL: {gatewayURL}")

## Step 6: Create Lambda Target with Campaign Analytics Tools

This section creates a gateway target that transforms our Lambda function into MCP-compliant tools with detailed schemas. The target defines three tools: player context retrieval with SQL filtering, semantic item recommendations, and dynamic pricing calculations, all using the GATEWAY_IAM_ROLE credential provider for secure access.

In [ ]:
# Configure Lambda target with campaign analytics tools
lambda_target_config = {
    "mcp": {
        "lambda": {
            "lambdaArn": lambda_function_arn,
            "toolSchema": {
                "inlinePayload": [
                    {
                        "name": "get_player_context_by_criteria",
                        "description": "Get comprehensive context for players matching SQL criteria across all datasets",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "criteria": {"type": "string", "description": "SQL WHERE clause to filter players (e.g., 'platform = \"PC\" AND churn_risk_score < 0.3')"},
                            },
                            "required": ["criteria"]
                        }
                    },
                    {
                        "name": "recommend_items_for_segment",
                        "description": "Search for items using semantic query based on segment analysis",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "query_text": {"type": "string", "description": "Search query for items (e.g., 'PC premium faction weapons', 'mobile casual consumables')"},
                                "campaign_goal": {"type": "string", "description": "Campaign objective: monetization, retention, engagement"}
                            },
                            "required": ["query_text"]
                        }
                    },
                    {
                        "name": "calculate_pricing_strategy",
                        "description": "Calculate final prices based on agent's pricing instructions - acts as a pricing calculator",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "items": {"type": "array", "description": "List of items to price with their base prices"},
                                "pricing_instructions": {"type": "array", "description": "Pricing instructions per item: [{\"type\": \"discount\", \"value\": 15}, {\"type\": \"markup\", \"value\": 20}]"}
                            },
                            "required": ["items", "pricing_instructions"]
                        }
                    }
                ]
            }
        }
    }
}

credential_config = [{"credentialProviderType": "GATEWAY_IAM_ROLE"}]
targetname = 'GameAnalyticsTools'

response = gateway_client.create_gateway_target(
    gatewayIdentifier=gatewayID,
    name=targetname,
    description='Game Analytics Tools with S3 Vectors Integration',
    targetConfiguration=lambda_target_config,
    credentialProviderConfigurations=credential_config
)

print("Lambda target created successfully!")
target_id = response['targetId']
print(f"Target ID: {target_id}")

## Step 7: Test the Gateway with MCP Client

This section demonstrates how to interact with our newly created MCP tools using the Strands Agents MCP client. We test each tool individually to verify functionality and demonstrate the complete workflow from Cognito authentication to tool execution with real campaign analytics scenarios.

In [ ]:
# Request access token from Cognito
print("Requesting access token from Amazon Cognito...")
token_response = gateway_utils.get_token(user_pool_id, client_id, client_secret, scopeString, REGION)
print(token_response)
token = token_response["access_token"]
print("Token acquired successfully!")

### MCP Client Setup

Here we configure the MCP client to communicate with our AgentCore Gateway using streamable HTTP transport with Bearer token authentication. The [MCPClient from Strands Agents](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/tools/mcp-tools/) provides convenient interfaces for tool discovery, invocation, and response handling. MCP is an open protocol that standardizes how applications provide context to Large Language Models, enabling agents to connect to external tools and services.

In [ ]:
# Setup MCP client for direct tool calls
from mcp.client.streamable_http import streamablehttp_client 
from strands.tools.mcp.mcp_client import MCPClient

def create_streamable_http_transport():
    return streamablehttp_client(gatewayURL, headers={"Authorization": f"Bearer {token}"})

client = MCPClient(create_streamable_http_transport)
print("MCP client setup complete!")

### Test 1: Player Context Retrieval

This test demonstrates the `get_player_context_by_criteria` tool using a complex SQL query to identify high-value PC players. The tool executes the query against datasets, then retrieves comprehensive player data from S3 Vectors, showcasing the power of combining structured SQL queries with semantic vector retrieval.

In [ ]:
import json

with client:
    # List available tools
    tools = client.list_tools_sync()
    print(f"Available tools: {[tool.tool_name for tool in tools]}")
    
    # Test 1: Get player context
    print("\n1. Getting player context...")
    result1 = client.call_tool_sync(
        tool_use_id="context-1",
        name=f"{targetname}___get_player_context_by_criteria",
        arguments={
            "criteria": "SELECT DISTINCT p.player_id FROM player_profiles p JOIN lifetime_spend s ON p.player_id = s.player_id WHERE p.platform = 'PC' AND p.engagement_intensity_score > 70 AND p.churn_risk_score < 0.3 AND s.total_spend_usd > 25"
        }
    )
    response_data = json.loads(result1['content'][0]['text'])
    body_data = json.loads(response_data['body'])
    print(json.dumps(body_data, indent=2))

### Test 2: Semantic Item Recommendations

This test showcases the `recommend_items_for_segment` tool's semantic search capabilities. Using the natural language query, the tool queries S3 Vectors for semantically similar items and returns top recommendations with relevance scores and metadata.

In [ ]:
with client:
    # Test 2: Recommend items
    print("\n2. Recommending items...")
    result2 = client.call_tool_sync(
        tool_use_id="items-1",
        name=f"{targetname}___recommend_items_for_segment",
        arguments={
            "query_text": "Corporate Alliance legendary epic weapons with high price and high player level requirement",
            "campaign_goal": "monetization"
        }
    )
    response_data = json.loads(result2['content'][0]['text'])
    body_data = json.loads(response_data['body'])
    print(json.dumps(body_data, indent=2))

### Test 3: Dynamic Pricing Strategy Calculation

This test demonstrates the `calculate_pricing_strategy` tool's flexibility in applying various pricing models. The tool processes items with base prices and applies discount, markup, or fixed price strategies, showing how AI agents can dynamically adjust pricing based on campaign requirements and market conditions.

In [ ]:
with client:
    # Test 3: Calculate pricing
    print("\n3. Calculating pricing strategy...")
    result3 = client.call_tool_sync(
        tool_use_id="pricing-1",
        name=f"{targetname}___calculate_pricing_strategy",
        arguments={
            "items": [{"item_id": "weapon_001", "name": "Test Weapon", "base_price": 10.0}],
            "pricing_instructions": [{"type": "discount", "value": 15}]
        }
    )
    response_data = json.loads(result3['content'][0]['text'])
    body_data = json.loads(response_data['body'])
    print(json.dumps(body_data, indent=2))

## Step 8: Save Gateway Configuration for Lab 5

This final step preserves our gateway configuration in AWS Systems Manager Parameter Store for Lab 5. The stored configuration includes gateway URL, Cognito authentication parameters, and OAuth scope definitions needed for MCP tool access.

In [ ]:
import json

ssm_client = boto3.client('ssm')

gateway_config = {
    "gateway_url": gatewayURL,
    "user_pool_id": user_pool_id,
    "client_id": client_id,
    "client_secret": client_secret,
    "scope_string": scopeString,
}

try:
    ssm_client.put_parameter(
        Name='/game-analytics/gateway-config',
        Value=json.dumps(gateway_config),
        Type='String',
        Overwrite=True,
        Description='Gateway config for Lab 5'
    )
    print("✅ Gateway configuration saved to SSM")
except Exception as e:
    print(f"❌ Failed to save to SSM: {e}")

## Congratulations! 🎉

You have successfully:

✅ **Created game analytics Lambda function** with S3 Vectors integration  
✅ **Set up AgentCore Gateway** with Cognito authentication  
✅ **Converted Lambda functions into MCP tools** with detailed schemas  
✅ **Tested tools** using Strands Agent with conversational and direct API calls  
✅ **Saved gateway configuration** to SSM Parameter Store for Lab 5

### Key Tools Created:
- **get_player_context_by_criteria**: Retrieves comprehensive player data across all datasets based on the segment criteria
- **recommend_items_for_segment**: Uses S3 Vectors to find optimal items for segments
- **calculate_pricing_strategy**: Optimizes pricing and discount strategies

### Key Concepts Learned:
- **AgentCore Gateway**: Transforms Lambda functions into MCP servers
- **MCP Protocol**: Standardized communication between AI agents and tools
- **Dual Authentication**: Inbound (Cognito JWT) and Outbound (IAM) security
- **Tool Schema Definition**: Structured input/output specifications for tools

This foundation enables the campaign agent to generate data-driven marketing campaigns with specialized MCP tools.